In [1]:
# check langchain version it should be 0.3 if not uncomment below pip command to install the correct version
import langchain, langchain_community
print(langchain.__version__)
print(langchain_community.__version__)

0.3.27
0.3.24


In [2]:
#!pip uninstall langchain -y && pip uninstall langchain-core -y && pip uninstall langchain-community -y
#!pip install langchain==0.3.27 langchain-openai==0.3.33 langchain-community==0.3.24


In [ ]:
# Pharma Agent 

In [3]:
# Check pharma_agent.md for description

from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import AgentExecutor, create_tool_calling_agent
from dotenv import load_dotenv

# Load environment variables
load_dotenv()
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [4]:
mock_database = {
    "aspirin": {"use": "Pain relief", "contraindications": "Ulcers, bleeding disorders"},
    "metformin": {"use": "Diabetes Type 2", "contraindications": "Severe kidney issues"},
    "ibuprofen": {"use": "Inflammation and fever", "contraindications": "Asthma, bleeding disorders"},
    "paracetamol": {"use": "Pain relief and fever", "contraindications": "Liver disease"},
    "amoxicillin": {"use": "Bacterial infections", "contraindications": "Penicillin allergy"},
    "atorvastatin": {"use": "Cholesterol reduction", "contraindications": "Liver disease"},
    "lisinopril": {"use": "Hypertension", "contraindications": "Pregnancy, angioedema history"},
    "omeprazole": {"use": "Acid reflux", "contraindications": "Liver disease"},
    "simvastatin": {"use": "Cholesterol reduction", "contraindications": "Liver disease"},
    "levothyroxine": {"use": "Hypothyroidism", "contraindications": "Untreated adrenal insufficiency"},
    "ciprofloxacin": {"use": "Bacterial infections", "contraindications": "Tendon disorders"},
    "clopidogrel": {"use": "Prevent blood clots", "contraindications": "Active bleeding"},
    "fluoxetine": {"use": "Depression", "contraindications": "MAOI use"},
    "warfarin": {"use": "Prevent blood clots", "contraindications": "Bleeding disorders"}
}

In [5]:
@tool
def lookup_drug(drug_name: str) -> str:
    """this would look up drug information from a database"""
    drug = mock_database.get(drug_name.lower())
    return drug if drug else "Drug not found."

@tool
def safety_check(drug_name: str) -> str:
    """this would check for contraindications"""
    drug = mock_database.get(drug_name.lower())
    if not drug:
        return "No safety data available."
    return f"Contraindications: {drug['contraindications']}"

@tool
def dosage_calculator(input_data: str) -> str:
    """
    Example: 'patient_weight=75 mg_per_kg=5'
    """
    values = dict(item.split("=") for item in input_data.split())
    dose = float(values["patient_weight"]) * float(values["mg_per_kg"])
    return f"Recommended dose: {dose} mg"

@tool
def external_medical_search(query: str) -> str:
    """this would call an external medical database API"""
    return f"(Simulated summary from PubMed/clinical API) About: {query}"
    # In real implementation, integrate with clinical trial APIs


tools = [lookup_drug, safety_check, dosage_calculator, external_medical_search]

In [ ]:
# Agent Prompt Template
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are an expert pharmaceutical assistant. Use the tools provided to answer pharmaceutical-related queries accurately.
                                          
User Request: {input},
("placeholder", "{agent_scratchpad}"
""")
# create agent using above tools
agent = create_tool_calling_agent(llm=model, tools=tools, prompt=prompt)
executor = AgentExecutor(agent=agent, tools=tools, verbose=True,max_iterations=1)
# test the agent with a complex query
output = executor.invoke({"input": "Provide information on ibuprofen, check its safety for a patient with asthma, calculate dosage for a 70kg patient at 10mg/kg, and summarize recent studies on its efficacy."})
print("Agent Output:", output)



> Entering new AgentExecutor chain...

Invoking: `lookup_drug` with `{'drug_name': 'ibuprofen'}`


{'use': 'Inflammation and fever', 'contraindications': 'Asthma, bleeding disorders'}
Invoking: `safety_check` with `{'drug_name': 'ibuprofen'}`


Contraindications: Asthma, bleeding disorders
Invoking: `dosage_calculator` with `{'input_data': 'patient_weight=70 mg_per_kg=10'}`


Recommended dose: 700.0 mg
Invoking: `external_medical_search` with `{'query': 'recent studies on ibuprofen efficacy'}`


(Simulated summary from PubMed/clinical API) About: recent studies on ibuprofen efficacy
Invoking: `lookup_drug` with `{'drug_name': 'ibuprofen'}`


{'use': 'Inflammation and fever', 'contraindications': 'Asthma, bleeding disorders'}
Invoking: `safety_check` with `{'drug_name': 'ibuprofen'}`


Contraindications: Asthma, bleeding disorders
Invoking: `dosage_calculator` with `{'input_data': 'patient_weight=70 mg_per_kg=10'}`


Recommended dose: 700.0 mg
Invoking: `external_medical_search` with 

In [ ]:
# sample test query
# "Provide information on ibuprofen, check its safety for a patient with asthma, calculate dosage for a 70kg patient at 10mg/kg, and summarize recent studies on its efficacy."
# "What are the uses and contraindications of metformin?"
# "What is the medicine for pain relief and what are its contraindications?"
# "What medicine recommend for a patient with diabetes type 2 who has kidney issues?"
# "For fever management, what drug would you suggest and what safety checks should be considered?"